In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score,\
                            accuracy_score, balanced_accuracy_score,classification_report,\
                            ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.model_selection import train_test_split

import lightgbm as lgb
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout, multiply, Concatenate
from tensorflow.keras.layers import BatchNormalization, Activation, Embedding, ZeroPadding2D, LeakyReLU
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import RandomNormal
import tensorflow.keras.backend as K
from sklearn.utils import shuffle


np.random.seed(1635848)

In [ ]:
class cGAN():
    
    """
    Class containing 3 methods (and __init__): generator, discriminator and train.
    Generator is trained using random noise and label as inputs. Discriminator is trained
    using real/fake samples and labels as inputs.
    """
    
    def __init__(self,latent_dim=32, out_shape=6):
        
        self.latent_dim = latent_dim
        self.out_shape = out_shape 
        self.num_classes = 2
        # using Adam as our optimizer
        optimizer = Adam(0.0002, 0.5)
        
        # building the discriminator
        self.discriminator = self.discriminator()
        self.discriminator.compile(loss=['binary_crossentropy'],
                                   optimizer=optimizer,
                                   metrics=['accuracy'])

        # building the generator
        self.generator = self.generator()

        noise = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,))
        gen_samples = self.generator([noise, label])
        
        # we don't train discriminator when training generator
        self.discriminator.trainable = False
        valid = self.discriminator([gen_samples, label])

        # combining both models
        self.combined = Model([noise, label], valid)
        self.combined.compile(loss=['binary_crossentropy'],
                              optimizer=optimizer,
                             metrics=['accuracy'])


    def generator(self):
        init = RandomNormal(mean=0.0, stddev=0.02)
        model = Sequential()

        model.add(Dense(128, input_dim=self.latent_dim))
        model.add(Dropout(0.2))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(256))
        model.add(Dropout(0.2))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(512))
        model.add(Dropout(0.2))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(self.out_shape, activation='tanh'))

        noise = Input(shape=(self.latent_dim,))
        label = Input(shape=(1,), dtype='int32')
        label_embedding = Flatten()(Embedding(self.num_classes, self.latent_dim)(label))
        
        model_input = multiply([noise, label_embedding])
        gen_sample = model(model_input)

        return Model([noise, label], gen_sample, name="Generator")

    
    def discriminator(self):
        init = RandomNormal(mean=0.0, stddev=0.02)
        model = Sequential()

        model.add(Dense(512, input_dim=self.out_shape, kernel_initializer=init))
        model.add(LeakyReLU(alpha=0.2))
        
        model.add(Dense(256, kernel_initializer=init))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        
        model.add(Dense(128, kernel_initializer=init))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))
        
        model.add(Dense(1, activation='sigmoid'))
        
        gen_sample = Input(shape=(self.out_shape,))
        label = Input(shape=(1,), dtype='int32')
        label_embedding = Flatten()(Embedding(self.num_classes, self.out_shape)(label))

        model_input = multiply([gen_sample, label_embedding])
        validity = model(model_input)

        return Model(inputs=[gen_sample, label], outputs=validity, name="Discriminator")


    def train(self, X_train, y_train, pos_index, neg_index, epochs, sampling=False, batch_size=32, sample_interval=100, plot=True): 
        
        # though not recommended, defining losses as global helps as in analysing our cgan out of the class
        global G_losses
        global D_losses
        
        G_losses = []
        D_losses = []
        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        for epoch in range(epochs):
            
            # if sampling==True --> train discriminator with 8 sample from postivite class and rest with negative class
            if sampling:
                idx1 = np.random.choice(pos_index, 8)
                idx0 = np.random.choice(neg_index, batch_size-8)
                idx = np.concatenate((idx1, idx0))
            # if sampling!=True --> train discriminator using random instances in batches of 32
            else:
                idx = np.random.choice(len(y_train), batch_size)
            samples, labels = X_train[idx], y_train[idx]
            samples, labels = shuffle(samples, labels)
            
            # Sample noise as generator input
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_samples = self.generator.predict([noise, labels])

            # label smoothing
            if epoch < epochs//1.5:
                valid_smooth = (valid+0.1)-(np.random.random(valid.shape)*0.1)
                fake_smooth = (fake-0.1)+(np.random.random(fake.shape)*0.1)
            else:
                valid_smooth = valid 
                fake_smooth = fake
                
            # Train the discriminator
            self.discriminator.trainable = True
            d_loss_real = self.discriminator.train_on_batch([samples, labels], valid_smooth)
            d_loss_fake = self.discriminator.train_on_batch([gen_samples, labels], fake_smooth)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            self.discriminator.trainable = False
            sampled_labels = np.random.randint(0, 2, batch_size).reshape(-1, 1)
            # Train the generator
            g_loss = self.combined.train_on_batch([noise, sampled_labels], valid)

            if (epoch+1)%sample_interval==0:
                print('[%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f'
                  % (epoch, epochs, d_loss[0], g_loss[0]))
            G_losses.append(g_loss[0])
            D_losses.append(d_loss[0])
            if plot:
                if epoch+1==epochs:
                    plt.figure(figsize=(10,5))
                    plt.title("Generator and Discriminator Loss")
                    plt.plot(G_losses,label="G")
                    plt.plot(D_losses,label="D")
                    plt.xlabel("iterations")
                    plt.ylabel("Loss")
                    plt.legend()
                    plt.show()

In [ ]:
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')
df.head()

In [ ]:
le = preprocessing.LabelEncoder()
for i in ['Timestamp_cubic','Vazao', 'Vazao_bbr', 'Atraso(ms)','Hop_count','Bottleneck', 'Link_bottleneck']:
    df[i] = le.fit_transform(df[i].astype(str))

In [ ]:
df.head()

In [ ]:
df = df[-2000:]
df.shape

In [ ]:
df['Vazao_bbr']

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

y = scaler.fit_transform(df[['Vazao_bbr']])

X = scaler.fit_transform(df.drop('Vazao_bbr', axis=1))


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
print(X_train, X_test, y_train, y_test)

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train)
y_pred = linreg.predict(X_test)

In [ ]:
print(y_pred)

In [ ]:
cgan = cGAN()


In [ ]:
y_train = y_train.reshape(-1,1)
pos_index = np.where(y_train==1)[0]
neg_index = np.where(y_train==0)[0]
cgan.train(X_train, y_train, pos_index, neg_index, epochs=500)

In [ ]:
# we want to generate 19758 instances with class value 0 since that represents how many 0s are in the label of the real training set
noise = np.random.normal(0, 1, (19758, 32))
sampled_labels = np.zeros(19758).reshape(-1, 1)


gen_samples = cgan.generator.predict([noise, sampled_labels])

gen_df = pd.DataFrame(data = gen_samples,
                      columns=df.drop('Vazao_bbr', axis=1).columns)

In [ ]:
noise_2 = np.random.normal(0, 1, (6290, 32))
sampled_labels_2 = np.ones(6290).reshape(-1, 1)


gen_samples_2 = cgan.generator.predict([noise_2, sampled_labels_2])

gen_df_2 = pd.DataFrame(data = gen_samples_2,
                     columns=df.drop('Vazao_bbr', axis=1).columns)

In [ ]:
gen_df_2['Vazao_bbr'] = 1
gen_df['Vazao_bbr']=0

df_gan = pd.concat([gen_df_2, gen_df], ignore_index=True, sort=False)
df_gan = df_gan.sample(frac=1).reset_index(drop=True)

X_train_2 = df_gan.drop('Vazao_bbr', axis=1)
y_train_2 = df_gan['Vazao_bbr'].values

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train_2, y_train_2)

y_pred = linreg.predict(X_test)


In [ ]:
y_pred

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras.layers import BatchNormalization
import tensorflow as tf
from sklearn.metrics import mean_squared_error

class GAN:
    def __init__(self, latent_dim=32, out_shape=1):
        self.latent_dim = latent_dim
        self.out_shape = out_shape

        optimizer = Adam(0.0002, 0.5)

        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(loss='binary_crossentropy',
                                   optimizer=optimizer,
                                   metrics=['accuracy'])

        # Build the generator
        self.generator = self.build_generator()

        # The generator takes noise as input and generates Vazao_bbr values
        z = Input(shape=(self.latent_dim,))
        gen_values = self.generator(z)

        # For the combined model we do not train the discriminator
        self.discriminator.trainable = False

        # The discriminator takes generated samples as input and determines validity
        validity = self.discriminator(gen_values)

        # The combined model (generator + discriminator)
        self.combined = Model(z, validity)
        self.combined.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    def build_generator(self):
        model = Sequential()
        model.add(Dense(128, input_dim=self.latent_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(256))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        # Output layer for Vazao_bbr: using 'tanh' assuming it's scaled to [-1, 1]
        model.add(Dense(self.out_shape, activation='tanh'))

        noise = Input(shape=(self.latent_dim,))
        gen_sample = model(noise)

        return Model(noise, gen_sample, name="Generator")

    def build_discriminator(self):
        model = Sequential()
        model.add(Dense(512, input_dim=self.out_shape))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(256))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(128))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(1, activation='sigmoid'))

        value_input = Input(shape=(self.out_shape,))
        validity = model(value_input)
        return Model(value_input, validity, name="Discriminator")

    def train(self, data, epochs=1000, batch_size=32, sample_interval=100):
        # data is assumed to be an array of Vazao_bbr values scaled to [-1, 1]
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        G_losses = []
        D_losses = []

        for epoch in range(epochs):
            # Train Discriminator
            idx = np.random.randint(0, data.shape[0], batch_size)
            real_samples = data[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_samples = self.generator.predict(noise)

            d_loss_real = self.discriminator.train_on_batch(real_samples, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_samples, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.combined.train_on_batch(noise, valid)

            if (epoch + 1) % sample_interval == 0:
                print(f"[{epoch+1}/{epochs}] D_loss: {d_loss[0]:.4f} G_loss: {g_loss[0]:.4f}")
            G_losses.append(g_loss[0])
            D_losses.append(d_loss[0])

        # Plot losses
        plt.figure(figsize=(10,5))
        plt.title("Generator and Discriminator Loss")
        plt.plot(G_losses, label="G")
        plt.plot(D_losses, label="D")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

# Exemplo de uso:
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')

df = df.drop(columns=['Link_bottleneck'])

# Supondo que Vazao_bbr seja a variável alvo contínua
data = df[['Vazao_bbr']].values.astype(float)

# Escalonamento para [-1, 1]
scaler = MinMaxScaler(feature_range=(-1,1))
data_scaled = scaler.fit_transform(data)

gan = GAN(latent_dim=32, out_shape=1)
gan.train(data_scaled, epochs=1000, batch_size=64, sample_interval=100)

# Após o treinamento do GAN, gerando dados sintéticos
noise = np.random.normal(0, 1, (1000, 32))
synthetic_vazao_bbr = gan.generator.predict(noise)
synthetic_vazao_bbr_rescaled = scaler.inverse_transform(synthetic_vazao_bbr)

# Agora, se quisermos avaliar um modelo de regressão (por exemplo, LinearRegression) com RMSE
# Dividindo dataset original em treino e teste
X = df.drop('Vazao_bbr', axis=1).values
Y = df['Vazao_bbr'].values
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

linreg = LinearRegression()
linreg.fit(X_train, y_train)
y_pred = linreg.predict(X_test)

# Cálculo do RMSE
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE: {rmse}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras.layers import BatchNormalization
import tensorflow as tf
from sklearn.metrics import mean_squared_error

class GAN:
    def __init__(self, latent_dim=32, out_shape=1):
        self.latent_dim = latent_dim
        self.out_shape = out_shape

        optimizer = Adam(0.0002, 0.5)

        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(loss='binary_crossentropy',
                                   optimizer=optimizer,
                                   metrics=['accuracy'])

        # Build the generator
        self.generator = self.build_generator()

        # The generator takes noise as input and generates Vazao_bbr values
        z = Input(shape=(self.latent_dim,))
        gen_values = self.generator(z)

        # For the combined model we do not train the discriminator
        self.discriminator.trainable = False

        # The discriminator takes generated samples as input and determines validity
        validity = self.discriminator(gen_values)

        # The combined model (generator + discriminator)
        self.combined = Model(z, validity)
        self.combined.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    def build_generator(self):
        model = Sequential()
        model.add(Dense(128, input_dim=self.latent_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(256))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        # Output layer for Vazao_bbr: using 'tanh' assuming it's scaled to [-1, 1]
        model.add(Dense(self.out_shape, activation='tanh'))

        noise = Input(shape=(self.latent_dim,))
        gen_sample = model(noise)

        return Model(noise, gen_sample, name="Generator")

    def build_discriminator(self):
        model = Sequential()
        model.add(Dense(512, input_dim=self.out_shape))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(256))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(128))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.4))

        model.add(Dense(1, activation='sigmoid'))

        value_input = Input(shape=(self.out_shape,))
        validity = model(value_input)
        return Model(value_input, validity, name="Discriminator")

    def train(self, data, epochs=1000, batch_size=32, sample_interval=100):
        # data is assumed to be an array of Vazao_bbr values scaled to [-1, 1]
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        G_losses = []
        D_losses = []

        for epoch in range(epochs):
            # Train Discriminator
            idx = np.random.randint(0, data.shape[0], batch_size)
            real_samples = data[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_samples = self.generator.predict(noise)

            d_loss_real = self.discriminator.train_on_batch(real_samples, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_samples, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.combined.train_on_batch(noise, valid)

            if (epoch + 1) % sample_interval == 0:
                print(f"[{epoch+1}/{epochs}] D_loss: {d_loss[0]:.4f} G_loss: {g_loss[0]:.4f}")
            G_losses.append(g_loss[0])
            D_losses.append(d_loss[0])

        # Plot losses
        plt.figure(figsize=(10,5))
        plt.title("Generator and Discriminator Loss")
        plt.plot(G_losses, label="G")
        plt.plot(D_losses, label="D")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')

df = df.drop(columns=['Link_bottleneck', 'Hop_count', 'Bottleneck'])

# Supondo que Vazao_bbr seja a variável alvo contínua
data = df[['Vazao_bbr']].values.astype(float)

# Escalonamento para [-1, 1]
scaler = MinMaxScaler(feature_range=(-1,1))
data_scaled = scaler.fit_transform(data)

gan = GAN(latent_dim=32, out_shape=3)
gan.train(data_scaled, epochs=1000, batch_size=64, sample_interval=100)

# Após o treinamento do GAN, gerando dados sintéticos
noise = np.random.normal(0, 1, (1000, 32))
synthetic_vazao_bbr = gan.generator.predict(noise)
synthetic_vazao_bbr_rescaled = scaler.inverse_transform(synthetic_vazao_bbr)

print(synthetic_vazao_bbr_rescaled)
# # Agora, se quisermos avaliar um modelo de regressão (por exemplo, LinearRegression) com RMSE
# # Dividindo dataset original em treino e teste
# X = df.drop('Vazao_bbr', axis=1).values
# Y = df['Vazao_bbr'].values
# X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

# linreg = LinearRegression()
# linreg.fit(X_train, y_train)
# y_pred = linreg.predict(X_test)

# # Cálculo do RMSE
# rmse = mean_squared_error(y_test, y_pred, squared=False)
# print(f"RMSE: {rmse}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Concatenate, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

#########################
# Criação do dataset de exemplo
#########################
np.random.seed(42)
N = 1000
df = pd.DataFrame({
    'atraso': np.random.rand(N)*100,
    'vazao_cubic': np.random.rand(N)*50,
    'vazao_bbr': np.random.rand(N)*50
})

# Divide 70% treino e 30% teste
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

X_train = train_df.values
X_test = test_df.values

#########################
# Remover 15% dos dados do TESTE
#########################
total_values = X_test.size
num_missing = int(total_values * 0.15)
missing_positions = np.random.choice(total_values, num_missing, replace=False)

# Guardar os valores verdadeiros antes de removê-los
X_test_flat = X_test.flatten()
Y_true_missing = X_test_flat[missing_positions]

# Criar versão com missing
X_test_missing = X_test.copy().flatten()
X_test_missing[missing_positions] = np.nan
X_test_missing = X_test_missing.reshape(X_test.shape)

# Máscara (1 para faltante, 0 para observado, conforme pedido)
M_missing = np.isnan(X_test_missing).astype(float)

# Imputação inicial por média
imputer = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer.fit_transform(X_test_missing)

# Normalizar dados do treino (para treinamento do GAN)
X_min = np.nanmin(X_train, axis=0)
X_max = np.nanmax(X_train, axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1

# Normaliza X_test_imputed_mean
X_test_norm = (X_test_imputed_mean - X_min) / X_range

# M para GAIN (no GAIN, M=1 para observado, 0 para faltante)
M = 1 - M_missing  # invertendo pois GAIN normalmente usa 1 para observado

#########################
# Definição do GAIN-like (Gerador e Discriminador)
#########################
dim = X_test.shape[1]
latent_dim = dim  # Pode ser ajustado

X_input = Input(shape=(dim,))
Z_input = Input(shape=(dim,))
M_input = Input(shape=(dim,))

# Gerador
G_input = Concatenate()([X_input, Z_input, M_input])
G_h = Dense(128)(G_input)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_h = Dense(128)(G_h)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_out = Dense(dim, activation='sigmoid')(G_h)

X_hat = Multiply()([M_input, X_input]) + Multiply()([(1 - M_input), G_out])
generator = Model([X_input, Z_input, M_input], X_hat)

# Discriminador
X_hat_input = Input(shape=(dim,))
D_input = Concatenate()([X_hat_input, M_input])
D_h = Dense(128)(D_input)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_h = Dense(128)(D_h)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_out = Dense(dim, activation='sigmoid')(D_h)

discriminator = Model([X_hat_input, M_input], D_out)

discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

discriminator.trainable = False
D_pred = discriminator([generator([X_input, Z_input, M_input]), M_input])
combined = Model([X_input, Z_input, M_input], D_pred)
combined.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

#########################
# Treinamento do GAN
#########################
X_data = X_test_norm
M_data = M
epochs = 1000
batch_size = 64

for epoch in range(epochs):
    idx = np.random.randint(0, X_data.shape[0], batch_size)
    X_batch = X_data[idx]
    M_batch = M_data[idx]
    Z_batch = np.random.uniform(0, 1, (batch_size, dim))
    
    # Gerar imputação
    X_hat_batch = generator.predict([X_batch, Z_batch, M_batch], verbose=0)
    
    # Labels para o discriminador: M_batch indica onde é observado (1 = real), 0 = imputado
    # Queremos D dizendo 1 para valores observados e 0 para imputados
    D_labels = M_batch
    
    d_loss = discriminator.train_on_batch([X_hat_batch, M_batch], D_labels)
    
    # Treino do gerador: quer "enganar" o D, fazendo-o prever 1 em tudo
    trick_labels = np.ones((batch_size, dim))
    g_loss = combined.train_on_batch([X_batch, Z_batch, M_batch], trick_labels)
    
    if (epoch+1) % 100 == 0:
        print(f"[{epoch+1}/{epochs}] d_loss: {d_loss:.4f}, g_loss: {g_loss:.4f}")

#########################
# Imputação final com o gerador treinado
#########################
Z_full = np.random.uniform(0,1,(X_data.shape[0], dim))
X_imputed_norm = generator.predict([X_data, Z_full, M_data])
X_imputed_final = X_imputed_norm * X_range + X_min

# Avaliar RMSE apenas nos dados faltantes
X_imputed_final_flat = X_imputed_final.flatten()
Y_pred_missing = X_imputed_final_flat[missing_positions]

rmse_gan_imputation = mean_squared_error(Y_true_missing, Y_pred_missing, squared=False)
print(f"RMSE da imputação com GAN: {rmse_gan_imputation:.4f}")

# Podemos também comparar com a imputação por média
X_mean_flat = X_test_imputed_mean.flatten()
Y_pred_missing_mean = X_mean_flat[missing_positions]
rmse_mean_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_mean, squared=False)
print(f"RMSE da imputação com média: {rmse_mean_imputation:.4f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Concatenate, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

#########################
# Criação do dataset de exemplo
#########################
np.random.seed(42)
N = 1000
df = pd.DataFrame({
    'atraso': np.random.rand(N)*100,
    'vazao_cubic': np.random.rand(N)*50,
    'vazao_bbr': np.random.rand(N)*50
})

# Divide 70% treino e 30% teste
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

X_train = train_df.values
X_test = test_df.values

#########################
# Remover 15% dos dados do TESTE
#########################
total_values = X_test.size
num_missing = int(total_values * 0.15)
missing_positions = np.random.choice(total_values, num_missing, replace=False)

# Guardar os valores verdadeiros antes de removê-los
X_test_flat = X_test.flatten()
Y_true_missing = X_test_flat[missing_positions]

# Criar versão com missing
X_test_missing = X_test.copy().flatten()
X_test_missing[missing_positions] = np.nan
X_test_missing = X_test_missing.reshape(X_test.shape)

# Máscara (1 para faltante, 0 para observado)
M_missing = np.isnan(X_test_missing).astype(float)

#########################
# Imputação por média
#########################
imputer_mean = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer_mean.fit_transform(X_test_missing)
X_mean_flat = X_test_imputed_mean.flatten()
Y_pred_missing_mean = X_mean_flat[missing_positions]
rmse_mean_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_mean, squared=False)
print(f"RMSE da imputação com média: {rmse_mean_imputation:.4f}")

#########################
# Imputação por KNN
#########################
imputer_knn = KNNImputer(n_neighbors=5)
X_test_imputed_knn = imputer_knn.fit_transform(X_test_missing)
X_knn_flat = X_test_imputed_knn.flatten()
Y_pred_missing_knn = X_knn_flat[missing_positions]
rmse_knn_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_knn, squared=False)
print(f"RMSE da imputação com KNN: {rmse_knn_imputation:.4f}")

#########################
# Preparação para o GAN (GAIN-like)
#########################
X_min = np.nanmin(X_train, axis=0)
X_max = np.nanmax(X_train, axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1

# Normalizar imputação inicial (por média) para servir de base ao GAIN
X_test_norm = (X_test_imputed_mean - X_min) / X_range

# M para GAIN (1 observado, 0 faltante)
M = 1 - M_missing

dim = X_test.shape[1]
latent_dim = dim

# Rede geradora
X_input = Input(shape=(dim,))
Z_input = Input(shape=(dim,))
M_input = Input(shape=(dim,))

G_input = Concatenate()([X_input, Z_input, M_input])
G_h = Dense(128)(G_input)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_h = Dense(128)(G_h)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_out = Dense(dim, activation='sigmoid')(G_h)
X_hat = Multiply()([M_input, X_input]) + Multiply()([(1 - M_input), G_out])
generator = Model([X_input, Z_input, M_input], X_hat)

# Rede discriminadora
X_hat_input = Input(shape=(dim,))
D_input = Concatenate()([X_hat_input, M_input])
D_h = Dense(128)(D_input)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_h = Dense(128)(D_h)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_out = Dense(dim, activation='sigmoid')(D_h)
discriminator = Model([X_hat_input, M_input], D_out)

discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

discriminator.trainable = False
D_pred = discriminator([generator([X_input, Z_input, M_input]), M_input])
combined = Model([X_input, Z_input, M_input], D_pred)
combined.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

#########################
# Treinamento do GAIN
#########################
X_data = X_test_norm
M_data = M
epochs = 1000
batch_size = 64

for epoch in range(epochs):
    idx = np.random.randint(0, X_data.shape[0], batch_size)
    X_batch = X_data[idx]
    M_batch = M_data[idx]
    Z_batch = np.random.uniform(0, 1, (batch_size, dim))
    
    # Gerar imputação
    X_hat_batch = generator.predict([X_batch, Z_batch, M_batch], verbose=0)
    
    # Labels para o discriminador: 1 = observado, 0 = faltante
    D_labels = M_batch
    d_loss = discriminator.train_on_batch([X_hat_batch, M_batch], D_labels)
    
    # Treino do gerador: quer enganar o discriminador
    trick_labels = np.ones((batch_size, dim))
    g_loss = combined.train_on_batch([X_batch, Z_batch, M_batch], trick_labels)
    
    if (epoch+1) % 100 == 0:
        print(f"[{epoch+1}/{epochs}] d_loss: {d_loss:.4f}, g_loss: {g_loss:.4f}")

#########################
# Imputação final com o gerador treinado (GAN)
#########################
Z_full = np.random.uniform(0,1,(X_data.shape[0], dim))
X_imputed_norm = generator.predict([X_data, Z_full, M_data])
X_imputed_final = X_imputed_norm * X_range + X_min

X_gan_flat = X_imputed_final.flatten()
Y_pred_missing_gan = X_gan_flat[missing_positions]
rmse_gan_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_gan, squared=False)
print(f"RMSE da imputação com GAN: {rmse_gan_imputation:.4f}")

#########################
# Comparação Final
#########################
print("\nComparação de RMSE nas imputações:")
print(f"Média: {rmse_mean_imputation:.4f}")
print(f"KNN: {rmse_knn_imputation:.4f}")
print(f"GAN: {rmse_gan_imputation:.4f}")


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Concatenate, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import tensorflow as tf

def weighted_bce(y_true, y_pred, mask, weight_factor=5.0):
    # Aqui usamos tf.keras.backend.binary_crossentropy
    # para obter a perda no formato [batch_size, n_features]
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = 1.0 + (weight_factor - 1.0)*(1 - mask)
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)


#########################
# Criação do dataset de exemplo
#########################
np.random.seed(42)
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')
df = df.drop(columns=['Timestamp_cubic', 'Hop_count', 'Bottleneck', 'Link_bottleneck'])
# Divide 70% treino e 30% teste
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

X_train = train_df.values
X_test = test_df.values

#########################
# Remover 15% dos dados do TESTE
#########################
total_values = X_test.size
num_missing = int(total_values * 0.15)
missing_positions = np.random.choice(total_values, num_missing, replace=False)

# Guardar os valores verdadeiros antes de removê-los
X_test_flat = X_test.flatten()
Y_true_missing = X_test_flat[missing_positions]

# Criar versão com missing
X_test_missing = X_test.copy().flatten()
X_test_missing[missing_positions] = np.nan
X_test_missing = X_test_missing.reshape(X_test.shape)

# Máscara (1 para faltante, 0 para observado)
M_missing = np.isnan(X_test_missing).astype(float)

#########################
# Imputação por média
#########################
imputer_mean = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer_mean.fit_transform(X_test_missing)
X_mean_flat = X_test_imputed_mean.flatten()
Y_pred_missing_mean = X_mean_flat[missing_positions]
rmse_mean_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_mean, squared=False)
print(f"RMSE da imputação com média: {rmse_mean_imputation:.4f}")

#########################
# Imputação por KNN
#########################
imputer_knn = KNNImputer(n_neighbors=5)
X_test_imputed_knn = imputer_knn.fit_transform(X_test_missing)
X_knn_flat = X_test_imputed_knn.flatten()
Y_pred_missing_knn = X_knn_flat[missing_positions]
rmse_knn_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_knn, squared=False)
print(f"RMSE da imputação com KNN: {rmse_knn_imputation:.4f}")

#########################
# Preparação para o GAN (GAIN-like)
#########################
X_min = np.nanmin(X_train, axis=0)
X_max = np.nanmax(X_train, axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1

# Normalizar imputação inicial (por média) para servir de base ao GAIN
X_test_norm = (X_test_imputed_mean - X_min) / X_range

# M para GAIN (1 observado, 0 faltante)
M = 1 - M_missing

dim = X_test.shape[1]
latent_dim = dim

# Rede geradora
X_input = Input(shape=(dim,))
Z_input = Input(shape=(dim,))
M_input = Input(shape=(dim,))

G_input = Concatenate()([X_input, Z_input, M_input])
G_h = Dense(128)(G_input)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_h = Dense(128)(G_h)
G_h = LeakyReLU(alpha=0.2)(G_h)
G_out = Dense(dim, activation='sigmoid')(G_h)
X_hat = Multiply()([M_input, X_input]) + Multiply()([(1 - M_input), G_out])
generator = Model([X_input, Z_input, M_input], X_hat)

# Rede discriminadora
X_hat_input = Input(shape=(dim,))
D_input = Concatenate()([X_hat_input, M_input])
D_h = Dense(128)(D_input)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_h = Dense(128)(D_h)
D_h = LeakyReLU(alpha=0.2)(D_h)
D_out = Dense(dim, activation='sigmoid')(D_h)
discriminator = Model([X_hat_input, M_input], D_out)

d_optimizer = Adam(0.0002, 0.5)
g_optimizer = Adam(0.0002, 0.5)

#########################
# Treinamento do GAIN
#########################
X_data = X_test_norm
M_data = M
epochs = 1000
batch_size = 64

trick_labels = np.ones((batch_size, dim))

for epoch in range(epochs):
    idx = np.random.randint(0, X_data.shape[0], batch_size)
    X_batch = X_data[idx]
    M_batch = M_data[idx]
    Z_batch = np.random.uniform(0, 1, (batch_size, dim))
    
    # Forward Geração
    X_hat_batch = generator.predict([X_batch, Z_batch, M_batch], verbose=0)
    
    # Labels para o discriminador: 1 = observado, 0 = faltante
    D_labels = M_batch.astype(np.float32)

    # Treino do discriminador com GradientTape
    with tf.GradientTape() as tape_d:
        D_pred_out = discriminator([X_hat_batch, M_batch], training=True)
        d_loss = weighted_bce(D_labels, D_pred_out, M_batch, weight_factor=5.0)
    d_grads = tape_d.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    
    # Treino do gerador
    # Queremos enganar o discriminador, então esperamos que o discriminador diga tudo=1
    with tf.GradientTape() as tape_g:
        G_pred_out = discriminator([generator([X_batch, Z_batch, M_batch], training=True), M_batch], training=True)
        g_loss = tf.keras.losses.binary_crossentropy(trick_labels, G_pred_out)
        g_loss = tf.reduce_mean(g_loss)
    g_grads = tape_g.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))
    
    if (epoch+1) % 100 == 0:
        print(f"[{epoch+1}/{epochs}] d_loss: {d_loss:.4f}, g_loss: {g_loss:.4f}")

#########################
# Imputação final com o gerador treinado (GAN)
#########################
Z_full = np.random.uniform(0,1,(X_data.shape[0], dim))
X_imputed_norm = generator.predict([X_data, Z_full, M_data])
X_imputed_final = X_imputed_norm * X_range + X_min

X_gan_flat = X_imputed_final.flatten()
Y_pred_missing_gan = X_gan_flat[missing_positions]
rmse_gan_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_gan, squared=False)
print(f"RMSE da imputação com GAN: {rmse_gan_imputation}")

#########################
# Comparação Final
#########################
print("\nComparação de RMSE nas imputações:")
print(f"Média: {rmse_mean_imputation}")
print(f"KNN: {rmse_knn_imputation}")
print(f"GAN: {rmse_gan_imputation}")


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Concatenate, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def weighted_bce(y_true, y_pred, mask, weight_factor=5.0):
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = 1.0 + (weight_factor - 1.0)*(1 - mask)
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)


# Carregar e preparar dados
np.random.seed(42)
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')
df = df.drop(columns=['Timestamp_cubic', 'Hop_count', 'Bottleneck', 'Link_bottleneck'])

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

X_train = train_df.values
X_test = test_df.values

total_values = X_test.size
num_missing = int(total_values * 0.15)
missing_positions = np.random.choice(total_values, num_missing, replace=False)

X_test_flat = X_test.flatten()
Y_true_missing = X_test_flat[missing_positions]

X_test_missing = X_test.copy().flatten()
X_test_missing[missing_positions] = np.nan
X_test_missing = X_test_missing.reshape(X_test.shape)

M_missing = np.isnan(X_test_missing).astype(float)

# Imputação média
imputer_mean = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer_mean.fit_transform(X_test_missing)
X_mean_flat = X_test_imputed_mean.flatten()
Y_pred_missing_mean = X_mean_flat[missing_positions]
rmse_mean_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_mean, squared=False)

# Imputação KNN
imputer_knn = KNNImputer(n_neighbors=5)
X_test_imputed_knn = imputer_knn.fit_transform(X_test_missing)
X_knn_flat = X_test_imputed_knn.flatten()
Y_pred_missing_knn = X_knn_flat[missing_positions]
rmse_knn_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_knn, squared=False)


X_min = np.nanmin(X_train, axis=0)
X_max = np.nanmax(X_train, axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1

X_test_norm = (X_test_imputed_mean - X_min) / X_range
M = 1 - M_missing
dim = X_test.shape[1]

def build_models(dim):
    # Gerador
    X_input = Input(shape=(dim,))
    Z_input = Input(shape=(dim,))
    M_input = Input(shape=(dim,))

    G_input = Concatenate()([X_input, Z_input, M_input])
    G_h = Dense(128)(G_input)
    G_h = LeakyReLU(alpha=0.2)(G_h)
    G_h = Dense(128)(G_h)
    G_h = LeakyReLU(alpha=0.2)(G_h)
    G_out = Dense(dim, activation='sigmoid')(G_h)
    X_hat = Multiply()([M_input, X_input]) + Multiply()([(1 - M_input), G_out])
    generator = Model([X_input, Z_input, M_input], X_hat)

    # Discriminador
    X_hat_input = Input(shape=(dim,))
    D_input = Concatenate()([X_hat_input, M_input])
    D_h = Dense(128)(D_input)
    D_h = LeakyReLU(alpha=0.2)(D_h)
    D_h = Dense(128)(D_h)
    D_h = LeakyReLU(alpha=0.2)(D_h)
    D_out = Dense(dim, activation='sigmoid')(D_h)
    discriminator = Model([X_hat_input, M_input], D_out)

    return generator, discriminator

# Definir parâmetros do grid search
weight_factors = [2.0, 5.0, 10.0] 
epochs_list = [500, 1000, 2000]

best_rmse = np.inf
best_params = None

for wf in weight_factors:
    for ep in epochs_list:
        print(f"\nTreinando com weight_factor={wf}, epochs={ep}")
        
        # Reconstruir os modelos e otimizadores a cada loop
        generator, discriminator = build_models(dim)
        d_optimizer = Adam(0.0002, 0.5)
        g_optimizer = Adam(0.0002, 0.5)

        X_data = X_test_norm
        M_data = M
        batch_size = 64
        trick_labels = np.ones((batch_size, dim))

        for epoch in range(ep):
            idx = np.random.randint(0, X_data.shape[0], batch_size)
            X_batch = X_data[idx]
            M_batch = M_data[idx]
            Z_batch = np.random.uniform(0, 1, (batch_size, dim))
            
            X_hat_batch = generator.predict([X_batch, Z_batch, M_batch], verbose=0)
            
            D_labels = M_batch.astype(np.float32)
            with tf.GradientTape() as tape_d:
                D_pred_out = discriminator([X_hat_batch, M_batch], training=True)
                d_loss = weighted_bce(D_labels, D_pred_out, M_batch, weight_factor=wf)
            d_grads = tape_d.gradient(d_loss, discriminator.trainable_variables)
            d_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))
            
            with tf.GradientTape() as tape_g:
                G_pred_out = discriminator([generator([X_batch, Z_batch, M_batch], training=True), M_batch], training=True)
                g_loss = tf.keras.losses.binary_crossentropy(trick_labels, G_pred_out)
                g_loss = tf.reduce_mean(g_loss)
            g_grads = tape_g.gradient(g_loss, generator.trainable_variables)
            g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))
            
            if (epoch+1) % 500 == 0:
                print(f"[{epoch+1}/{ep}] d_loss: {d_loss:.4f}, g_loss: {g_loss:.4f}")

        # Após treinar, imputar com o gerador atual
        Z_full = np.random.uniform(0,1,(X_data.shape[0], dim))
        X_imputed_norm = generator.predict([X_data, Z_full, M_data])
        X_imputed_final = X_imputed_norm * X_range + X_min

        X_gan_flat = X_imputed_final.flatten()
        Y_pred_missing_gan = X_gan_flat[missing_positions]
        rmse_gan_imputation = mean_squared_error(Y_true_missing, Y_pred_missing_gan, squared=False)

        print(f"Resultado RMSE da GAN com weight_factor={wf}, epochs={ep}: {rmse_gan_imputation}")

        if rmse_gan_imputation < best_rmse:
            best_rmse = rmse_gan_imputation
            best_params = (wf, ep)

# Imprimir o melhor resultado
print("\nMelhor resultado para GAN:")
print(f"RMSE: {best_rmse}, com weight_factor={best_params[0]} e epochs={best_params[1]}")

# Comparação final
print("\nComparação de RMSE nas imputações:")
print(f"Média: {rmse_mean_imputation}")
print(f"KNN: {rmse_knn_imputation}")
print(f"GAN (Melhor achado no grid search): {best_rmse}")
